# 第18课 · 信息丢了还能还原吗？——可逆性与秩（rank）、零空间（null space）与奇异矩阵诊断

**目标**：判断一个方阵是否可逆的三条判据，以及「严格对角占优」这一快速充分条件。

**为什么对 Aurora 重要**：Jacobi/Gauss-Seidel 迭代求解器以对角占优为收敛前提；`is_sdd` 检查是求解器入口的第一道关卡。

← **上一课**　[L17 · 特征分解 A=PDP⁻¹](L17_eigen_diagonalization.ipynb)

> 上节课学习了 **特征分解 A=PDP⁻¹**：换坐标系让矩阵乘法变成标量乘法。  
> 本课将探讨 **可逆性与秩**。

## 本课剧情：信息有没有丢失？

想象你把一段语音压缩成 MP3。有损压缩 = 信息丢失 = 不可逆。  
想象你对信号做 FFT。FFT 是可逆的——有 ifft 可以完美还原。

矩阵也有同样的区分：
- **可逆矩阵**：把每个输入映射到唯一输出；知道输出，就能还原输入。信息没有丢失。
- **奇异矩阵（singular）**：把多个不同输入压缩到同一输出，无法区分；信息丢失了。

判断可逆性有三条等价充要条件：

| 判据 | 工具 | 计算复杂度 |
|---|---|---|
| `det(A) ≠ 0` | `np.linalg.det` | O(n³) |
| 所有特征值 ≠ 0 | `np.linalg.eigvals` | O(n³) |
| 严格对角占优（SDD）| 逐行检查 | O(n²)——快！ |

本课实现 `is_sdd(A)`，这是迭代求解器（Jacobi/Gauss-Seidel）的入口卫士：只有 SDD 矩阵才能保证收敛。

## 1. 三条等价判据

**判据 1：det(A) ≠ 0**（充要）  
行列式等于零 ↔ 矩阵把空间"压扁"了一个维度 ↔ 信息丢失 ↔ 不可逆。

**判据 2：所有特征值 ≠ 0**（充要）  
特征值 = 各方向的缩放因子。有一个方向缩放为 0 → 那个方向的信息丢失 → 不可逆。

**判据 3：严格对角占优（SDD）**（充分条件，非充要）  
对每行 i：`|A[i,i]| > Σ_{j≠i} |A[i,j]|`  
也就是：主对角元素的绝对值 > 该行其他所有元素绝对值之和。  
SDD → 可逆（但可逆矩阵不一定 SDD）。

**手算例子**：A = [[3, 1], [1, 2]]
- 行 0：|3| > |1| ✓
- 行 1：|2| > |1| ✓
→ SDD，矩阵可逆。

**反例**：A = [[1, 2], [2, 4]]（行线性相关，det=0，不可逆，且非SDD）

## 符号入口：先看形状，再看运算

本节处理方阵 `A`（shape `(n, n)`），核心工具是 `np.linalg.det(A)` 和 `np.linalg.eigvals(A)`。判断可逆性其实就看两个数字：一个行列式标量和一组特征值向量（vector）。

In [ ]:
import numpy as np
A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)
print('det =', round(np.linalg.det(A)))
print('特征值:', np.round(np.linalg.eigvals(A), 3))
print('两条充要判据都说：可逆 =', abs(np.linalg.det(A))>1e-9)

## 动手观察：从 det 和特征值读可逆性

运行下面的代码，注意 `det` 接近零时特征值中是否出现接近零的分量——两条充要判据在数值上指向同一个事实。

In [ ]:
import numpy as np

# 秩 = 线性无关行（列）的数目
mats = [
    ('满秩', np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]])),
    ('秩2', np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])),
    ('秩1', np.array([[1.,2.],[2.,4.]])),
]
for name, A in mats:
    r = np.linalg.matrix_rank(A)
    inv_ok = (r == min(A.shape))
    print(f'{name}  shape={A.shape}  rank={r}  可逆={inv_ok}')


## 代码实验：观察矩阵如何作用于多个向量

可逆矩阵把不同的输入映射到不同的输出；奇异矩阵则会把两个不同向量压缩到同一个方向。

In [ ]:
import numpy as np

# 零空间（核）= SVD 中对应零奇异值的右奇异向量
A = np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])
U, s, Vt = np.linalg.svd(A)
print(f'奇异值 σ = {np.round(s, 6)}')
# 零奇异值对应 Vt 最后一行
null_vec = Vt[-1]
print(f'零空间向量 n = {np.round(null_vec, 4)}')
print(f'A @ n = {np.round(A @ null_vec, 10)}  （应为全零）')


## ✏️ 练习：提取零空间向量并验证

奇异矩阵的零空间（null space）不止是「存在」，还可以用 SVD 精确提取。

**任务**：给定秩亏矩阵 `B`，找到零空间向量 `v`，使得 `B @ v ≈ 0`。

**推理路线**：
1. 对 `B` 做 SVD：`U, s, Vt = np.linalg.svd(B)`
2. 最小奇异值对应 `Vt` 的最后一行，即零空间向量。
3. 如果该奇异值 < 1e-9，则 `B` 实际上秩亏，`Vt[-1]` 是零空间向量。
4. 验证：`‖B @ v‖ < 1e-8`。

In [ ]:
import numpy as np

def extract_null_vector(B):
    """返回矩阵 B 的（最小奇异值对应）零空间向量 v。
    若 B 是满秩矩阵，返回最接近零空间的向量（最小奇异值对应方向）。
    """
    B = np.asarray(B, float)
    # ✏️ TODO: 用 SVD 提取零空间向量
    # 提示：np.linalg.svd(B) 返回 U, s, Vt；
    #       最小奇异值对应 Vt 的最后一行。
    raise NotImplementedError("TODO: implement extract_null_vector — 用 np.linalg.svd(B) 提取 Vt[-1]")

# 测试：秩亏矩阵（rank 2，有真实零空间）
B = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])

try:
    v = extract_null_vector(B)
    residual = np.linalg.norm(B @ v)
    print(f'零空间向量 v = {np.round(v, 6)}')
    print(f'‖B @ v‖ = {residual:.2e}  （应 < 1e-8，即数值零）')
    assert residual < 1e-8, f'残差过大：{residual:.2e}，请检查实现'
    print('✅ 零空间向量验证通过！')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  {e}')
    print('请先完成 extract_null_vector 的 TODO 实现，再运行此单元格。')


## 2. ✏️ 实现 `is_sdd(A)`（严格对角占优）

对每行 i：`|A[i,i]| > Σ_{j≠i} |A[i,j]|` 是否都成立。

**推理路线**：
1. 取第 i 行的对角元绝对值：`d = abs(A[i, i])`。
2. 取第 i 行所有元素绝对值之和：`s = sum(abs(A[i]))`；非对角部分之和为 `s - d`（无需单独遍历 j≠i）。
3. 检查 `d > s - d` 对所有行成立；用 `all(...)` 把 n 个布尔值合并为一个返回值。

**参考输入输出**：`A=[[4,1],[1,3]]` → 行0：4>1 ✓；行1：3>1 ✓ → `True`；`B=[[1,2],[2,1]]` → 行0：1<2 ✗ → `False`

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写 `is_sdd` 前明确三件事

- 输入：`A`，shape `(n, n)` 的方阵
- 关键步骤：对每行 `i`，比较 `abs(A[i,i])` 与 `sum(abs(A[i])) - abs(A[i,i])`
- 返回：`bool`，所有行均满足则为 `True`

In [ ]:
def is_sdd(A):
    A = np.asarray(A, float)
    # ✏️ TODO: 返回是否严格对角占优 (True/False)
    raise NotImplementedError("TODO: implement is_sdd — 参考推理路线：d = abs(A[i,i])，s = sum(abs(A[i]))，检查 d > s - d")


In [ ]:
# 三条可逆性判据验证例题
M1 = np.array([[2,0,1],[1,4,2],[1,3,6]], float)   # s.d.d. 且可逆
M2 = np.array([[1,0,2],[2,5,1],[3,2,13]], float)  # 非 s.d.d. 但仍可逆
M3 = np.array([[1,1],[1,1]], float)               # 非 s.d.d. 且不可逆
try:
    for name, M in [('M1',M1),('M2',M2),('M3',M3)]:
        print(f'{name}: sdd={is_sdd(M)}, 可逆={abs(np.linalg.det(M))>1e-9}')
    assert is_sdd(M1) and not is_sdd(M2) and not is_sdd(M3)
    print('\n✅ 与课件三例一致。注意 M2 说明：sdd 是充分非必要条件。')
except (NotImplementedError, TypeError):
    print('⬜ 请先完成上方 is_sdd 的 TODO 实现，再运行此验证单元格。')

**🔗 Aurora 连接**：`is_sdd` 是 Aurora 迭代求解器的入口卫士——系数矩阵只有通过 s.d.d. 检查，Jacobi/Gauss-Seidel 求解流程才会启动；此时谱半径（spectral radius） < 1 有数学保证，迭代误差按几何级数衰减而非发散。

**补充例题对应**：三条可逆性判据 + 严格对角占优。

**线代前导章节完成**：L09–L18 覆盖向量、矩阵、方程组、行列式、特征值与可逆性。

## 🎨 图示：严格对角占优矩阵(对角线显著占优 ⇒ 可逆)

In [ ]:
from aurora.laviz import style, heatmap
style()
heatmap([[2,0,1],[1,4,2],[1,3,6]], title='严格对角占优 ⇒ 可逆', cmap='magma');

In [ ]:
import numpy as np

# 参数实验：矩阵退化实验——让一列线性依赖于其他列
print(f"{'依赖系数 c':>12}  {'rank':>6}  {'det':>12}  {'可逆':>6}")
for c in [0.0, 0.5, 1.0, 1.5, 2.0]:
    A = np.array([[1.,2.,c],[0.,1.,1.],[0.,0.,c-1.]])
    r = np.linalg.matrix_rank(A)
    d = np.linalg.det(A)
    print(f'{c:>12.1f}  {r:>6d}  {d:>+12.4f}  {"是" if r==3 else "否":>6s}')


## 参数实验：Jacobi 迭代的收敛与发散

构造一个对角元明显大于同行其余元素绝对值之和的矩阵（如 `A = [[10,1,1],[1,10,1],[1,1,10]]`），用 Jacobi 迭代求解线性系统 `Ax = b`，打印每轮残差 `‖Ax_k - b‖`，确认误差逐步缩小至收敛。

再把对角元缩小使矩阵不再满足 s.d.d.（如 `A = [[1,2,2],[2,1,2],[2,2,1]]`），重跑相同迭代步数，观察残差不减反增或振荡。两次实验对比说明：`is_sdd` 通过是 Jacobi 收敛的理论保证，不是可有可无的布尔标记。

In [ ]:
import numpy as np

def jacobi(A, b, n_iter=30):
    """Jacobi 迭代：x_new[i] = (b[i] - Σ_{j≠i} A[i,j]·x[j]) / A[i,i]"""
    x = np.zeros_like(b, dtype=float)
    D_inv = 1.0 / np.diag(A)           # 对角元倒数
    residuals = []
    for _ in range(n_iter):
        x = D_inv * (b - (A @ x - np.diag(np.diag(A)) @ x))
        residuals.append(np.linalg.norm(A @ x - b))
    return x, residuals

# ① s.d.d. 矩阵 → 收敛
A_sdd = np.array([[10.,1.,1.],[1.,10.,1.],[1.,1.,10.]])
b_sdd = np.array([12.,12.,12.])
x_sdd, res_sdd = jacobi(A_sdd, b_sdd)
print('s.d.d. 最终残差:', f'{res_sdd[-1]:.2e}  (应趋近 0)')
assert res_sdd[-1] < 1e-8, 'Jacobi 在 s.d.d. 矩阵上应收敛'

# ② 非 s.d.d. 矩阵 → 发散
A_bad = np.array([[1.,2.,2.],[2.,1.,2.],[2.,2.,1.]])
b_bad = np.ones(3)
_, res_bad = jacobi(A_bad, b_bad)
print('非 s.d.d. 最终残差:', f'{res_bad[-1]:.2e}  (期望 >> 0，发散)')
print('✅ Jacobi 收敛 vs 发散对比完成')

## 本课收束

现在能用 `is_sdd(A)` 逐行检查严格对角占优，用 `np.linalg.det` 和 `np.linalg.eigvals` 验证两条充要判据。`is_sdd` 对应 Aurora 迭代求解器的入口检查：通过它，Jacobi 迭代才能保证收敛。下一课：**L19** 矩阵变换可视化——把线性变换的几何效果画出来，直观理解矩阵对空间的作用。

In [ ]:
# Aurora 连接：可逆矩阵在 DSP 中的应用 — DFT 矩阵是酉矩阵（行列式=1，列正交）
from aurora.audio.transforms import fft, ifft
import numpy as np
x = np.random.randn(8)
assert np.allclose(ifft(fft(x)), x, atol=1e-10), "ifft(fft(x)) ≠ x — DFT 矩阵不可逆"
print("✅ DFT 矩阵可逆性验证：ifft(fft(x)) ≈ x（误差 < 1e-10）")

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：可逆性手算（目标 8 分钟）

盖上屏幕，纸上作答：

**问 1**：A = [[2, 1], [1, 3]]，手算 det(A)（公式 ad−bc），A 可逆吗？

**问 2**：B = [[3, 1], [1, 2]]，逐行检查 is_sdd(B)：  
- 行 0：|3| > |1|？  
- 行 1：|2| > |1|？  
B 是 SDD 矩阵吗？

**问 3**：C = [[1, 2], [2, 4]]，det(C) = ? 它可逆吗？

**问 4**：A = [[10, 1, 1], [1, 10, 1], [1, 1, 10]]，逐行检查 is_sdd(A)：  
各行：|10| > |1|+|1| = 2？→ 这是 SDD 矩阵吗？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：det([[2,1],[1,3]])
A1 = np.array([[2., 1.], [1., 3.]])
det1 = float(np.linalg.det(A1))
assert np.isclose(det1, 5.0, atol=1e-10)
print(f"Q1 ✅  det([[2,1],[1,3]]) = 2×3-1×1 = {det1:.1f}  → 可逆")

# 问2：is_sdd([[3,1],[1,2]])
B = np.array([[3., 1.], [1., 2.]])
try:
    sdd_B = is_sdd(B)
    assert sdd_B, "[[3,1],[1,2]] 应为 SDD"
    print(f"Q2 ✅  [[3,1],[1,2]] 是 SDD：|3|>|1|, |2|>|1|  → 可逆")
except (NotImplementedError, TypeError):
    print("⬜ Q2：请先实现 is_sdd()，再运行对答案格")

# 问3：singular [[1,2],[2,4]]
C = np.array([[1., 2.], [2., 4.]])
det_C = float(np.linalg.det(C))
assert abs(det_C) < 1e-10, f"det 应≈0，得到 {det_C}"
print(f"\nQ3 ✅  det([[1,2],[2,4]]) = {det_C:.1e} ≈ 0  → 奇异，不可逆")

# 问4：is_sdd([[10,1,1],[1,10,1],[1,1,10]])
A4 = np.array([[10., 1., 1.], [1., 10., 1.], [1., 1., 10.]])
try:
    sdd4 = is_sdd(A4)
    assert sdd4
    print(f"Q4 ✅  diag-dominant(10): 每行 |10|>|1|+|1|=2  → SDD 矩阵")
except (NotImplementedError, TypeError):
    rows_ok4 = [abs(A4[i,i]) > sum(abs(A4[i,j]) for j in range(3) if j!=i) for i in range(3)]
    assert all(rows_ok4)
    print(f"Q4 ✅  每行 |10|>2，is_sdd=True  (is_sdd 待实现)")
print("\n🎉 可逆性白板挑战通过！det/eigenvalue/SDD 三条判据已内化。")

In [ ]:
# ✏️ 本课自评
l18_review = {
    "is_sdd_implemented":         None,  # is_sdd 实现并通过断言？True/False
    "three_invertibility_criteria": None, # 能背出三条可逆性判据？True/False
    "null_space_understood":       None,  # 理解零空间 = 被压缩到0的方向？True/False
    "sdd_implies_invertible":      None,  # 理解 SDD → 可逆（充分非充要）？True/False
    "whiteboard_passed":           None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l18_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l18_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L18 全部通关！进入 L19：矩阵变换图解')

---

→ **下一课**　[L19 · 矩阵变换图解](L19_visual_multiply.ipynb)

> 下节课将学习 **矩阵变换图解**：旋转、缩放、剪切与「列的线性组合」的视觉演示。